In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Dataset (same as before)
data = {
    "Size (sq.ft)": [1500, 2000, 2500, 1800, 2200],
    "Number of Bedrooms": [3, 4, 3, 2, 4],
    "Age of the House (years)": [20, 15, 10, 25, 5],
    "Price (in ₹ lakhs)": [70, 90, 110, 65, 120],
}
df = pd.DataFrame(data)
print(df)

In [ ]:
# Prepare the data
X = df[["Size (sq.ft)", "Number of Bedrooms", "Age of the House (years)"]].values
y = df["Price (in ₹ lakhs)"].values.reshape(-1, 1)#1d to 2d ,slice into numpy array
print("Features (X):")
print(X)
print("Features (y):")
print(y)


In [ ]:
# Standardize the features manually -scaling
mean_X = X.mean(axis=0)
std_X = X.std(axis=0)

In [ ]:
# Standardize the features (X_scaled = (X - mean) / std)
X_scaled = (X - mean_X) / std_X
print("standardized features(X):")
print(X_scaled)

In [ ]:
# Add bias term (intercept) to the feature matrix
X_bias = np.hstack((np.ones((X_scaled.shape[0], 1)), X_scaled))  # Adds a column of ones for the bias term
print("features with bias term(X_bias): ")
print(X_bias)

In [ ]:
#  question 1: Solving using Singular Value Decomposition (SVD)
def solve_svd(X, y):

    """

    Solve linear regression using Singular Value Decomposition (SVD)
    
    Parameters:
    - X: Feature matrix (with bias term)
    - y: Target vector (Price)
    
    Returns:
    - beta: Learned coefficients

    """
    # Compute the SVD of the feature matrix X
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    
    # Compute the pseudoinverse of X using SVD
    S_inv = np.diag(1 / S)
    X_pseudo = Vt.T @ S_inv @ U.T
    
    # Calculate the coefficients (beta)
    beta = X_pseudo @ y
    return beta


# Apply SVD to find the coefficients
beta_svd = solve_svd(X_bias, y)


# Print the coefficients using SVD
print(f"singular value decomposition Coefficients: {beta_svd.flatten()}")


# Predicted prices using SVD
y_pred_svd = X_bias @ beta_svd


# Compare predicted prices with actual prices (SVD)
predicted_vs_actual_svd = pd.DataFrame({
    "Actual Prices (Y)": y.flatten(),
    "Predicted Prices (Y_pred)": y_pred_svd.flatten(),
    "Difference": (y.flatten() - y_pred_svd.flatten())
})


In [ ]:
print("\nPredicted vs Actual Prices using SVD:")
print(predicted_vs_actual_svd)

In [ ]:
# question 2: Solving using Gradient Descent 
def gradient_descent(X, y, lr=0.001, iterations=2000, tol=1e-6):

    
    """
    
    Parameters:
    - X: Feature matrix
    - y: Target vector
    - lr: Learning rate 
    - iterations: Number of iterations 

    Returns:
    - beta: Learned coefficients
    - cost_history: List of cost values over iterations

    """

    m, n = X.shape  # Number of samples and features
    # Initialize coefficients (theta) to small random values
    beta = np.random.randn(n, 1) * 0.1  # Initialize with small random values
    cost_history = []

    previous_cost = float('inf')  # To track the cost for early stopping
    
    for i in range(iterations):
        # Predictions
        y_pred = X @ beta  # Predicting the prices
        
        # Compute cost (MSE)
        cost = (1 / (2 * m)) * np.sum((y_pred - y) ** 2)
        cost_history.append(cost)
        
        # Check for early stopping if the cost does not improve significantly
        if abs(previous_cost - cost) < tol:
            print(f"Stopping early at iteration {i} because cost change is below tolerance.")
            break
        previous_cost = cost
        
        # Compute gradients
        gradients = (1 / m) * (X.T @ (y_pred - y))  # Gradient of the cost function
        
        # Update coefficients (theta)
        beta -= lr * gradients

    return beta, cost_history


'''
roll number of helon 4
roll number of vivin 18
mean_roll = 11  # Mean roll number

Calculate hyperparameters
learning_rate = 0.01 + mean_roll / 10000  # lr= 0.0111
iterations = 100 + mean_roll  # 111 iterations

'''


# Apply Gradient Descent with a smaller learning rate and more iterations
beta_gd, cost_history = gradient_descent(X_bias, y, lr=0.0111, iterations=111)
print(f"Gradient Descent Coefficients: {beta_gd}")



# Predicted prices using gradient descent
y_pred_gd = X_bias @ beta_gd  # Predicting using the updated coefficients


# Compare predicted prices with actual prices (Gradient Descent)
predicted_vs_actual_gd = pd.DataFrame({
    "Actual Prices (Y)": y.flatten(),
    "Predicted Prices (Y_pred)": y_pred_gd.flatten(),
    "Difference": (y.flatten() - y_pred_gd.flatten())
})


In [ ]:
print("\nPredicted vs Actual Prices using Gradient Descent:")
print(predicted_vs_actual_gd)

In [ ]:
# Convergence plot for Gradient Descent (Cost vs. Iterations)
plt.figure(figsize=(10, 5))
plt.plot(cost_history, label='Cost (MSE) over iterations')
plt.xlabel('Iterations')
plt.ylabel('Cost (Mean Squared Error)')
plt.title('Convergence Plot for Gradient Descent')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Comparison of Actual vs Predicted Prices for both methods
predicted_vs_actual_comparison = pd.DataFrame({
    "Actual Prices (Y)": y.flatten(),
    "Predicted Prices (SVD)": y_pred_svd.flatten(),
    "Predicted Prices (GD)": y_pred_gd.flatten()
})

print("\nPredicted vs Actual Prices using SVD and Gradient Descent:")
print(predicted_vs_actual_comparison)

In [ ]:
# Plotting Actual vs Predicted Prices for both SVD and Gradient Descent
plt.figure(figsize=(10, 5))
plt.plot(predicted_vs_actual_comparison["Actual Prices (Y)"], label='Actual Prices', marker='o', linestyle='-', color='b')
plt.plot(predicted_vs_actual_comparison["Predicted Prices (SVD)"], label='Predicted Prices (SVD)', marker='x', linestyle='--', color='r')
plt.plot(predicted_vs_actual_comparison["Predicted Prices (GD)"], label='Predicted Prices (GD)', marker='x', linestyle='-.', color='g')
plt.xlabel('Index')
plt.ylabel('Price (in ₹ lakhs)')
plt.title('Comparison of Actual vs Predicted Prices (SVD and GD)')
plt.legend()
plt.grid(True)
plt.show()